In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.checkpoint")

In [ ]:
# 1. Загружаем и готовим базу
df = pd.read_parquet('/kaggle/input/datasets/mitchell11/fin-news/prepared_dataset.parquet')
df['datetime'] = pd.to_datetime(df['datetime'])

THRESHOLD = 0.01
df['target'] = (np.abs(df['return']) > THRESHOLD).astype(int)

# 2. Очистка текста
# Берем тикер + первые ~500 символов новости
df['text_input'] = "Тикер: " + df['ticker'] + ". Новость: " + df['text'].str[:500]

# Удаляем дубликаты
df = df.drop_duplicates(subset=['text_input'])

# 3. Идеальный баланс (Undersampling)
df_pos = df[df['target'] == 1]
df_neg = df[df['target'] == 0].sample(len(df_pos), random_state=42) # Берем столько же нулей
df_balanced = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Размер датасета после балансировки: {len(df_balanced)}")
print(df_balanced['target'].value_counts())

# 4. Сплит (теперь можно обычный, т.к. мы уже сбалансировали и перемешали)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced['text_input'].tolist(), 
    df_balanced['target'].tolist(), 
    test_size=0.15, 
    random_state=42
)

Размер датасета после балансировки: 1086
target
0    543
1    543
Name: count, dtype: int64


In [3]:
import pandas as pd
import numpy as np
import torch
import joblib
from transformers import AutoTokenizer, AutoModel
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Загружаем токенизатор и модель BERT
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
bert_model.eval()

# 2. Функция для батч-генерации эмбеддингов
def get_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=256, return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            output = bert_model(**encoded)
        
        # Используем CLS-токен
        cls_emb = output.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls_emb)
    
    return np.array(embeddings)

print("Генерация эмбеддингов для train...")
X_train_emb = get_embeddings(train_texts)
print("Генерация эмбеддингов для test...")
X_test_emb = get_embeddings(test_texts)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Генерация эмбеддингов для train...
Генерация эмбеддингов для test...


In [4]:
# ---------------------------------------------------------
# БЕЙЗЛАЙН 1: Logistic Regression
# ---------------------------------------------------------
print("\n--- Обучение Logistic Regression ---")
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_emb, train_labels)

preds_lr = model_lr.predict(X_test_emb)
print("Результаты LogReg:")
print(f"Accuracy: {accuracy_score(test_labels, preds_lr):.4f}")
print(f"F1 Macro: {f1_score(test_labels, preds_lr, average='macro'):.4f}")
print(classification_report(test_labels, preds_lr))

# Сохранение весов LogReg
joblib.dump(model_lr, 'logreg_baseline.pkl')


--- Обучение Logistic Regression ---
Результаты LogReg:
Accuracy: 0.5767
F1 Macro: 0.5757
              precision    recall  f1-score   support

           0       0.59      0.52      0.55        82
           1       0.57      0.63      0.60        81

    accuracy                           0.58       163
   macro avg       0.58      0.58      0.58       163
weighted avg       0.58      0.58      0.58       163



['logreg_baseline.pkl']

In [5]:
# ---------------------------------------------------------
# БЕЙЗЛАЙН 2: CatBoost
# ---------------------------------------------------------
print("\n--- Обучение CatBoost ---")
model_cb = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.03,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

# Используем eval_set для контроля переобучения
model_cb.fit(
    X_train_emb, train_labels, 
    eval_set=(X_test_emb, test_labels), 
    early_stopping_rounds=50
)

preds_cb = model_cb.predict(X_test_emb)
print("\nРезультаты CatBoost:")
print(f"Accuracy: {accuracy_score(test_labels, preds_cb):.4f}")
print(f"F1 Macro: {f1_score(test_labels, preds_cb, average='macro'):.4f}")
print(classification_report(test_labels, preds_cb))

# Сохранение модели CatBoost
model_cb.save_model('catboost_baseline.cbm')


--- Обучение CatBoost ---
0:	learn: 0.6262189	test: 0.5092025	best: 0.5092025 (0)	total: 107ms	remaining: 53.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5705521472
bestIteration = 38

Shrink model to first 39 iterations.

Результаты CatBoost:
Accuracy: 0.5706
F1 Macro: 0.5698
              precision    recall  f1-score   support

           0       0.58      0.52      0.55        82
           1       0.56      0.62      0.59        81

    accuracy                           0.57       163
   macro avg       0.57      0.57      0.57       163
weighted avg       0.57      0.57      0.57       163



In [6]:
!pip install -q transformers peft bitsandbytes datasets trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 29.4 MB/s eta 0:00:00


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score

# 1. Готовим Dataset HuggingFace
train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels})

model_id = "Qwen/Qwen2.5-1.5B"

# 2. Токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Для Qwen обязательно

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_tokenized = train_dataset.map(tokenize_fn, batched=True)
test_tokenized = test_dataset.map(tokenize_fn, batched=True)

# 3. Квантование (4-bit) для ускорения
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 4. Загрузка LLM как классификатора
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

# 5. Настройка LoRA (обучаем только ~1% параметров)
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    task_type="SEQ_CLS",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 6. Метрики
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/163 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,092,608 || all params: 1,544,809,984 || trainable%: 0.0707


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 7. Обучение
training_args = TrainingArguments(
    output_dir="./llm_finetuned",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.train()

# 8. Сохранение адаптеров
trainer.model.save_pretrained("./my_finance_llm")
tokenizer.save_pretrained("./my_finance_llm")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.033581,0.964652,0.509202,0.500919
2,0.676749,0.885265,0.576687,0.576432


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


('./my_finance_llm/tokenizer_config.json',
 './my_finance_llm/chat_template.jinja',
 './my_finance_llm/tokenizer.json')

In [11]:
import shutil
shutil.make_archive('virtual_documents', 'zip', '/kaggle/working/.virtual_documents')
shutil.make_archive('catboost_info', 'zip', '/kaggle/working/catboost_info')
shutil.make_archive('llm_finetuned', 'zip', '/kaggle/working/llm_finetuned')
shutil.make_archive('my_finance_llm', 'zip', '/kaggle/working/my_finance_llm')

'/kaggle/working/my_finance_llm.zip'